# Chapter 4: Collaborative Evaluation
How to measure whether two or more people label things the same way. **No LLM needed** in this notebook.

In [ ]:
from common import load_data
from collections import Counter
import numpy as np

data = load_data()["ch4"]
a, b, c3 = data["annotator_a"], data["annotator_b"], data["annotator_c"]
print("Annotator A:", a)
print("Annotator B:", b)

## 1. Percent agreement (simple but misleading)

In [ ]:
def percent_agreement(r1, r2):
    return sum(x == y for x, y in zip(r1, r2)) / len(r1)

print(f"A vs B percent agreement: {percent_agreement(a, b):.0%}")

## 2. Cohen's kappa (2 annotators, removes luck)
`kappa = (Po - Pe) / (1 - Pe)`
- `Po` = observed agreement
- `Pe` = agreement expected by pure chance

In [ ]:
def cohens_kappa(r1, r2):
    n = len(r1)
    po = sum(x == y for x, y in zip(r1, r2)) / n              # observed
    c1, c2 = Counter(r1), Counter(r2)
    pe = sum((c1[l] / n) * (c2[l] / n) for l in set(r1) | set(r2))   # by chance
    return (po - pe) / (1 - pe)

print(f"A vs B kappa: {cohens_kappa(a, b):.2f}")

### Check with the book example
100 summaries: both Pass = 75, A Pass/B Fail = 10, A Fail/B Pass = 5, both Fail = 10. Book answer: **kappa ≈ 0.48**.

In [ ]:
r1 = ["Pass"] * 75 + ["Pass"] * 10 + ["Fail"] * 5 + ["Fail"] * 10
r2 = ["Pass"] * 75 + ["Fail"] * 10 + ["Pass"] * 5 + ["Fail"] * 10

print(f"Percent agreement: {percent_agreement(r1, r2):.2f}")
print(f"Kappa            : {cohens_kappa(r1, r2):.2f}")

## 3. Why percent agreement can lie
95 of 100 traces are Pass. The two annotators disagree on **every** Fail.

In [ ]:
x = ["Pass"] * 95 + ["Fail"] * 5          # annotator X
y = ["Fail"] * 5 + ["Pass"] * 95          # annotator Y (fails different traces)

print(f"Percent agreement: {percent_agreement(x, y):.0%}   <- looks great")
print(f"Kappa            : {cohens_kappa(x, y):.2f}   <- reality: no real agreement")

## 4. Compare with scikit-learn (should match)

In [ ]:
from sklearn.metrics import cohen_kappa_score
print("Ours   :", round(cohens_kappa(a, b), 3))
print("sklearn:", round(cohen_kappa_score(a, b), 3))

## 5. Weighted kappa (ordered scores 1-5)
Being 1 point off is better than 4 points off, so we use weights.

In [ ]:
sa, sb = data["scores_a"], data["scores_b"]
print("Plain kappa    :", round(cohen_kappa_score(sa, sb), 2))
print("Weighted kappa :", round(cohen_kappa_score(sa, sb, weights="quadratic"), 2))

## 6. Fleiss' kappa (3 or more annotators)

In [ ]:
def fleiss_kappa(ratings):
    """ratings = list of items; each item = list of labels from all annotators"""
    labels = sorted({x for item in ratings for x in item})
    n_raters, n_items = len(ratings[0]), len(ratings)
    counts = np.array([[item.count(l) for l in labels] for item in ratings])
    p_label = counts.sum(axis=0) / (n_items * n_raters)
    p_item = ((counts ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    pe = (p_label ** 2).sum()
    return (p_item.mean() - pe) / (1 - pe)

items = list(zip(a, b, c3))                  # each item = (A label, B label, C label)
items = [list(i) for i in items]
print(f"Fleiss kappa (A, B, C): {fleiss_kappa(items):.2f}")

## 7. Read the number + find disagreements for the alignment session

In [ ]:
def interpret(k):
    if k < 0: return "poor"
    if k <= 0.20: return "slight"
    if k <= 0.40: return "fair"
    if k <= 0.60: return "moderate"
    if k <= 0.80: return "substantial"
    return "almost perfect"

k = cohens_kappa(a, b)
print(f"kappa = {k:.2f} -> {interpret(k)}  (aim for 0.6 or more)")

print("\nTraces to discuss (A and B disagree):")
for i, (x, y) in enumerate(zip(a, b), start=1):
    if x != y:
        print(f"  trace {i}: A={x}, B={y}")